# Design ClosingAid Service

<br>
## 1.0 Executive Summary & Core Objective

The `ClosingAid` service is an automated accounting bridge designed to streamline the onboarding of newly acquired rental properties. The service ingests unstructured real estate closing documents (ALTA, HUD-1, or settlement statement table images/PDFs), extracts the line items using Optical Character Recognition (OCR), and maps them into a balanced, structured Draft Manual Journal Entry.

Furthermore, the system isolates capitalized acquisition costs to
- calculate the property's Adjusted Cost Basis,
- automatically calculates the depreciable Building vs. Land Split,
- accurately attributes both cash-to-close requirements and
- external member-paid escrow deposits to individual partner capital accounts within a Multi-Member LLC structure.

The `ClosingAid` is a specialized skill and service integrated into the `llcRentalTracker` app to handle journaling of the closing statements into the financial books. 

## 1.1 llcRentalTracker Objective

Develop a ClosingAid to 
1. assist an operator in importing a `Property Closing Statement` (image)
2. assist an operating in entering transactions into the llcAsset ledger (json DB) from a `Property Closing Statement` 
3. display a `Closing Statement Balance Sheet` that correctly reflects the Assets, Equity (member) and Liabilities based solely from the `Property Closing Statement`.
4. display the Property Basis of the property based on the `Property Closing Statement`

## 1.2 Closing Aid Accounting Guidelines

Refere to llcRentalTracker.docs.design_BUS_01.5_ClosingAid.md file for design guidelines for the ClosingAid guide.  Special notice should be given to the contols specified by `3.1 Line-by-Line Tax Classification Rules` and the example `Tax Treatment column` on classification of line items. 

Note also section 4.1 and the ability to post 1 sided entries within the llcAsset.

## 2.0 Business Files

**llcRentalTracker Files:**
- This notebook: ~/GDrive/dev/pyTrackers/llcRentalTracker/20250820-ClosingToLedger.ipynb

**LLC-WBGroup Files:**
- Business Top Folder (BusTop) : ~/GDrive/Family/Assets/LLC-WBGroup
- Closing_PDF : $BusTop/Assets/805HighMesa/Docs/Final Closing Package (Buyer or Borrower)_2.pdf
- Closing Statement: table on page 3 of $Closing_PDF
- llcAsset DB: ~/GDrive/Family/Assets/LLC-WBGroup/books/2025/Accts/llcAssets.json


------------------------------
## 3.0 Scenario

### 3.1 Preface - Execute Closing, prior to journaling
1. Member Invests in Purchase of Property
    - member Pays Escros and Option Fees ($5000, $300) - outside of LLC Bank
    - member transfers Funds for purchase ($219000) into LLC Bank (see Bank llcExpRev.json ledger)
4. Execute Closing Statements at title company
    - Member Pays balance from funds in the LLC Bank (see Bank llcExpRev.json ledger)
    - closing statement table is converted into a dataframe via a notebook - see `Sample Raw Property Closing Statement` below

### 3.2 Journal Closing Statement into the Business llcAsset ledger (json DB). 
1. Login to llcRentalTracker app @ website
2. Operator goes to llcAssets View
3. Operator selects "ClosingAid" button
    - operator pastes closing statement into ClosingAid dialog
    - Closing Aid asks upfront questions to fillin common fields needed for step across all entries
        - closingDate - date of the closing 
        - closingDoc - select path to the documents provided by the title company
        - landPct - ask for the Assessors, current year, percent value of land :: total assessed value
        - assetType - "H" (House Rental) or "R" (RV Rental)
        - assetState - 'active', 'inactive', 'InConstruction', 'other' --> acctSub
        - tID_Prefix - prefix to use for every entry, "<assetType><dt>"
             - actual transaction tID will be <tID_Prefix>_<basis>_<assetType><propName>
        - propNm, - Short Name : assign `propNm` with <assetType>_<ShortName>
        - propAddrpropOwners - "[{memberID : Percent},...]

          ____ the following are auto generated ___
        - propID - <tID_Prefix>_transaction sequence number}
        - tDB, refDB - llcAsset
        - refDoc - f"{propNm}, Closing Docs, <taxBucketClassification>, <closingDoc>
4. ClosingAid converts the closing statement into dataframe showing Description, Debit, Credit, Cls
    - Classification is an aggregate index of various line item descriptions
5. ClosingAid identifies the `acct` for each non-null line item
    - Uses skill as an expert accountant in real estate
    - Ignore line items if Debit and Credit are both null/0
    - Use the COA (Chart of Accounts)
6. ClosingAid identifies the `dual account`to pu
    - ie. counter account to the `acct` to be associated with the line item,
    - uses the [description, Cls] and COA
    - puts the dual account name  into a `Ledger` column 
7. displays the following:
    - a. Display Closing Statement transaction summary
        - shows updated closing statement dataframe/table
        -  with added `acct` and `Ledger` account names
    - b. Display a balance sheet using the closing statement transactions, ie. line items
        - converts the dual accounting closing ledger (ie. table from a.) into a single account GL format
        - Uses the single account GL table to generate a ClosingBalanceSheet
        - The closing must adhere to Balance Sheet rule : Assets = Equity + Liabilities
    - c. Displays a table showing 'Property Basis` 
        - use IRS and accounting expert skills to compute the Property Basis
8. submit will create transactions in llcAssets, save to DB and reload view


---

## 4.0 AI: What are all the things needed

## 4.1 GL toGL() Changes

- The mapping of dual account ledgers (llcAsset, llcExpRev, etc.) need to be enhanced.
- Add  the feature that if the `Ledger` account equal "nan" then that side of the dual accounts is not added to the GL.
- The GL will only show the `acct` transaction.
- This allows special handling for entries such as closing statements where line items are posted with only 1 account entry.
- This is important per section 1.2 Accounting Guidelines where individual transaction entries are posted to the GL with only 1 side because the collection of all post reconcile with transactions outside the llcAsset for example. 

**Files to change — both GL expansion paths need the nan-Ledger guard:**

| File | Function | Change |
|---|---|---|
| `ledger/ledgerDB.py` | `toGL()` lines 203–216 | After building `dfL`, filter to rows where source `Ledger` was not NaN/null/string-'nan' before the concat |
| `ledger/stmtGL.py` | `toDoubleEntry()` line 209 | Replace `(r.get('Ledger') or '').strip()` with explicit guard: skip e2 when Ledger is `None`, `float('nan')`, or the string `'nan'` — current code fails because `float('nan')` is truthy |

**Validation**: A record with `Ledger = nan` produces exactly 1 GL row (the `acct` side). A record with a real `Ledger` value still produces 2 GL rows. Existing double-entry behavior is unchanged.

---

### 4.2 New Module — `ledger/closingAid.py`

Core `ClosingAid` class. **Each closing line item posts as a 1-sided entry (`Ledger = None`)** — per §4.1 and §1.2 accounting guidelines, the collection of all rows forms a balanced compound journal entry (ΣDebits = ΣCredits); no row-level dual account is needed.

**Four methods:**

| Method | Input | Output |
|---|---|---|
| `classify(rows)` | raw closing rows `{Description, Debit, Credit, Cls}` | same rows + `acct`, `Ledger=None`, `aType`, `amt`, `tax_bucket`; null/zero rows + Totals row dropped |
| `toBalanceSheet(classified)` | classified rows | `{Debit_total, Credit_total, balanced, delta}` |
| `propertyBasis(classified)` | classified rows | `{components: [{component, acct, amt, tax_bucket}], total_basis}` |
| `toAssetRecords(classified, preface)` | classified rows + preface dict (§3.2 step 3) | list of dicts in llcAssets DB record schema, with land/building split applied when `landPct > 0` |

**Three-bucket classifier** (per `design_BUS_01.5_ClosingAid.md §3.1`):

| Tax Bucket | Rule | Description keyword / Cls | aType | acct | Ledger |
|---|---|---|---|---|---|
| **Capitalize** | Adds to property basis (IRS Pub 551) | Sale Price | Debit | `Acct.Fixed.Tangible.InService` | None |
| **Capitalize** | Adds to property basis | Title Settlement / Closing Fee | Debit | `Acct.Fixed.Tangible.InService` | None |
| **Capitalize** | Adds to property basis | Recording fees / E-Recording / Govt Recording | Debit | `Acct.Fixed.Tangible.InService` | None |
| **Capitalize** | Adds to property basis | County tax / Property tax proration (Credit from seller) | Credit | `Acct.Fixed.Tangible.InService` | None |
| **Capitalize** | LLC-funded deposit applied to purchase | Deposit or Earnest / Earnest Money (LLC bank) | Credit | `Acct.Cash.Bank` | None |
| **Capitalize** | Member out-of-pocket escrow → member capital | Option Money (personal) | Credit | `Acct.Equity.Owner.Capital.Funds` | None |
| **Capitalize** | Net cash from LLC bank at closing | Cash-to-Close / Balance Due | Credit | `Acct.Cash.Bank` | None |
| **Amortize** | Loan fees amortized over loan term | Loan Origination / Origination Fee / Points / Appraisal | Debit | `Acct.Liab.Morgage` | None |
| **Expense** | Immediate deductible — hits P&L | HOA (any) / Homeowners Association (any) | Debit or Credit | `Acct.Exp.Operating` | None |
| **Skip** | — | Description "Totals" / "Total" OR both Debit and Credit null/0 | — | — | — |

**`toAssetRecords()` produces:**
- `tID = {tID_Prefix}_{seq+1:02d}` per row
- `propID = tID_Prefix` (property-level ID, same for all rows from this closing)
- `refDoc = f"{propNm}, Closing Docs, {tax_bucket}, {closingDoc}"` — tax_bucket embedded per row
- `Ledger = None` (1-sided entry per §4.1)
- `acctSub = assetState` from preface
- `tDB = refDB = 'llcAssets'`
- `dt` normalized to `YYYY.MM.DD` format from HTML date input (`YYYY-MM-DD`)

**Land/Building split (applied inside `toAssetRecords()` when `preface.landPct > 0`):**
- Collect all Capitalize + Debit + `Acct.Fixed.Tangible.InService` rows; sum = `total_basis`
- Replace them with two summary records:
  - Land: `total_basis × landPct/100` → `Acct.Fixed.Land`, Debit, Capitalize
  - Building: `total_basis × (1 − landPct/100)` → `Acct.Fixed.Tangible.InService`, Debit, Capitalize
- All other rows (Credit rows, Expense, Amortize) pass through unchanged

**Compound entry balance invariant**: `sum(Debit amts) == sum(Credit amts)` ± $0.01. Raise `ClosingBalanceError` if not balanced; dialog shows discrepancy.

---

### 4.3 New UI Module — `ui/llcClosingAid.py`

Defines `bind_closing_routes(app, objects, sanitize)` — called at the end of `llcMgmt._bind_routes()`:

| Route | Method | Payload | Purpose |
|---|---|---|---|
| `/api/closing/classify` | POST | `{rows: [...]}` | Return classified ledger rows + `tax_bucket` per row |
| `/api/closing/balance_sheet` | POST | `{rows: [...]}` | Return `{Debit_total, Credit_total, balanced, delta}` |
| `/api/closing/property_basis` | POST | `{rows: [...]}` | Return `{components, total_basis}` |
| `/api/closing/commit` | POST | `{rows: [...], preface: {...}}` | `toAssetRecords()` then `mgr.save(existing + records)`; return `{ok, count, tIDs}` |

The `preface` dict carries: `closingDate`, `closingDoc`, `landPct`, `assetType`, `assetState`, `tID_Prefix`, `propNm`, `propAddr`, `propOwners`.

---

### 4.4 New Template — `ui/templates/_closing_aid_dialog.html`

Multi-step modal dialog following `_aid_dialog.html` pattern. Steps map to §3.2:

| Step | Label | Fields / Content | Gate to proceed |
|---|---|---|---|
| **0 — Preface** | Common Fields | `closingDate` (date), `closingDoc` (text), `landPct` (number 0–100), `assetType` (H/R), `assetState` (dropdown), `tID_Prefix` (text), `propNm` (text), `propAddr` (text), `propOwners` (textarea JSON) | All required fields non-empty |
| **1 — Input** | Paste Statement | Textarea for raw closing statement (CSV/tab); "Parse" → `/api/closing/classify` | At least 1 classified row returned |
| **2 — Review Ledger** | Classified Rows | Editable table: Description, Debit/Credit amount, Tax Bucket badge (color-coded), `acct` dropdown (COA); `Ledger` hidden (always None) | — |
| **3 — Balance Check** | Compound Entry | ΣDebits vs ΣCredits; green "BALANCED" or red "OFF BY $X.XX" | Must show BALANCED to enable Commit |
| **4 — Property Basis** | Basis Breakdown | Read-only table by tax bucket; Total Property Basis; land/building split preview if `landPct > 0` | — |
| **Action bar** | | "Cancel" · "← Back" · "Next →" · (Step 4 only) "Commit to Assets Ledger" | Commit disabled when imbalanced |

Included from `table_view.html` only when `OBJ_TYPE == 'llcAssets'`. "ClosingAid" button added to the existing `toolbar-row`.

---

### 4.5 Land/Building Split

`landPct` is collected in the **Step 0 Preface form** (not post-commit). When `landPct > 0`, `toAssetRecords()` consolidates all Capitalize-Debit-InService rows into two records (Land + Building split) before writing to the DB. This is Phase 1 (at commit time), not post-commit.

---

### 4.6 Modifications to Existing Files

| File | Change |
|---|---|
| `ledger/ledgerDB.py` | `toGL()`: filter `dfL` to exclude rows where source `Ledger` is NaN/null/string-'nan' (see §4.1) |
| `ledger/stmtGL.py` | `toDoubleEntry()`: harden `ledger_acct` nan check (see §4.1) |
| `ui/llcMgmt.py` | Import `bind_closing_routes` from `ui.llcClosingAid`; call at end of `_bind_routes()` |
| `ui/templates/table_view.html` | Add "ClosingAid" button in `toolbar-row` when `obj_type == 'llcAssets'`; include `_closing_aid_dialog.html` at bottom of content block |

---

### 4.7 Clean Existing `llcAssets_WBGroupLLC.json` Entries

**Current state**: 8 stored records — `tID: a20250826-Escrow` (amt: 5300, Credit) is a literal duplicate (appears twice).

**Issues to fix:**
1. **Remove duplicate**: delete one of the two `a20250826-Escrow` entries; 7 unique records remain
2. **Existing entries use `Ledger ≠ null`** — correct for pre-ClosingAid dual-entry format; the §4.1 fix leaves these unchanged; new ClosingAid entries use `Ledger = null`; both styles coexist

---

### 4.8 Tests — `tests/test_closingAid.py`

| Test | Assertion |
|---|---|
| `test_toGL_nan_ledger_single_entry` | Record with `Ledger=np.nan` → exactly 1 GL row from `ledgerDB.toGL()` |
| `test_toDoubleEntry_nan_ledger` | `toDoubleEntry()` with `Ledger=float('nan')` → 1 row, no AttributeError |
| `test_classify_assigns_coa` | All non-null rows in `ccDict` get valid `acct` + `Ledger=None` |
| `test_classify_skips_null_and_totals` | Rows with both Debit/Credit null/0 AND the "Totals" row are excluded |
| `test_classify_three_buckets` | Every classified row carries `tax_bucket` in `{Capitalize, Amortize, Expense}` |
| `test_balance_sheet_balanced` | `toBalanceSheet(ccDict_classified)` → `balanced=True` ± $0.01 |
| `test_property_basis_components` | Sale Price + Title + Recording fees in components; HOA Transfer excluded |
| `test_to_asset_records_schema` | Output records have `dt`, `desc`, `amt`, `aType`, `acct`, `Ledger`, `tID`, `tDB`, `propNm`, `propID`, `acctSub`, `refDoc` |
| `test_to_asset_records_tid_prefix` | All `tID` values start with `tID_Prefix` from preface |
| `test_to_asset_records_refDoc_has_bucket` | Every record's `refDoc` contains its `tax_bucket` label |
| `test_land_split_two_records` | When `landPct=20`, InService-debit rows are replaced by `Acct.Fixed.Land` + `Acct.Fixed.Tangible.InService` records; compound balance unchanged |

---

### 4.9 Data Flow Summary

```
Step 0 — Operator fills Preface form
  closingDate, closingDoc, landPct, assetType, assetState,
  tID_Prefix, propNm, propAddr, propOwners
  → auto-set: propID = tID_Prefix, tDB = refDB = 'llcAssets'
  → refDoc template: f"{propNm}, Closing Docs, {tax_bucket}, {closingDoc}"
        ↓
Step 1 — Paste Raw Closing Statement (CSV / tab rows)
        ↓
POST /api/closing/classify
  ClosingAid.classify(rows)
  • drop: both Debit+Credit null/0, OR Description == 'Totals'
  • assign acct, aType, amt, tax_bucket per 3-bucket rules (§4.2)
  • set Ledger = None on every row (1-sided compound entry per §4.1)
        ↓
Step 2 — Operator reviews/corrects acct per row
        ↓
POST /api/closing/balance_sheet
  • sum Debit rows; sum Credit rows
  • return {Debit_total, Credit_total, balanced, delta}
  (Commit disabled until balanced)
        ↓
POST /api/closing/property_basis
  • filter tax_bucket == Capitalize, aType == Debit
  • label: purchase price / title fees / recording fees / tax proration
  • sum → Total Property Basis
  • if landPct > 0: show land/building split preview
        ↓
Step 3 — Operator clicks "Commit to Assets Ledger"
POST /api/closing/commit  {rows, preface}
  ClosingAid.toAssetRecords(classified, preface)
  • if landPct > 0: _apply_land_split() → replace InService-debit rows with
    Land record (landPct%) + Building record ((1-landPct)%)
  • per row: tID = {tID_Prefix}_{seq+1:02d}
             refDoc = f"{propNm}, Closing Docs, {tax_bucket}, {closingDoc}"
             Ledger = None  (1-sided per §4.1)
  → mgr.save(existing + records) → llcAssets working file
  → dialog closes, table_view reloads
```


## Sample RAW Property Closing Statement

- this code was generated by a AI chat request
- does not have `acct` nor `Ledger` accounts

In [83]:
import pandas as pd
import numpy as np

ccDict = [{'Description': 'Sale Price of Property',
  'Debit': 220000.0, 
  'Credit': np.nan,
  'Cls': 'Deposit'},
 {'Description': 'Deposit or Earnest Money from W&B Group, LLC',
  'Debit': np.nan,
  'Credit': 5000.0,
  'Cls': 'Deposit'},
 {'Description': 'Option Money from W&B Group, LLC',
  'Debit': np.nan,
  'Credit': 300.0,
  'Cls': 'Deposit'},
 {'Description': 'County taxes 1/1/2025 to 8/25/2025 @ $2,568.37/Year',
  'Debit': np.nan,
  'Credit': 1660.64,
  'Cls': 'Prorations'},
 {'Description': 'Homeowners Association Dues 8/25/2025 to 1/1/2026 @ $100.00/Year',
  'Debit': 35.34,
  'Credit': np.nan,
  'Cls': 'Prorations'},
 {'Description': "Title - Lender's coverage Premium $0.00 to Texas National Title, Inc.",
  'Debit': np.nan,
  'Credit': np.nan,
  'Cls': 'Titel Charges'},
 {'Description': 'Title - Settlement or closing fee $1,250.00 to Texas National Title, Inc.',
  'Debit': 625.0,
  'Credit': np.nan,
  'Cls': 'Titel Charges'},
 {'Description': 'Title - E-Recording Service Fee to Texas National Title, Inc.',
  'Debit': 8.0,
  'Credit': np.nan,
  'Cls': 'Titel Charges'},
 {'Description': 'Government Recording and Transfer Charges',
  'Debit': np.nan,
  'Credit': np.nan,
  'Cls': 'Titel Charges'},
 {'Description': 'Recording fees: Deed $29.25',
  'Debit': 29.25,
  'Credit': np.nan,
  'Cls': 'Titel Charges'},
 {'Description': 'Additional Settlement Charges',
  'Debit': np.nan,
  'Credit': np.nan,
  'Cls': 'Titel Charges'},
 {'Description': 'HOA Transfer Fees to Cedar Oak Mesa Property Owners Association',
  'Debit': 100.0,
  'Credit': np.nan,
  'Cls': "Add'l Charges"},
 {'Description': 'Totals',
  'Debit': 220897.59,
  'Credit': 6960.64,
  'Cls': "Add'l Charges"}]
df = pd.DataFrame.from_dict(ccDict)
c

In [84]:
print(df.set_index(['Cls','Description']).fillna('').to_string())

                                                                                             Buyer   Seller
Cls           Description                                                                                  
Deposit       Sale Price of Property                                                      220000.0         
              Deposit or Earnest Money from W&B Group, LLC                                           5000.0
              Option Money from W&B Group, LLC                                                        300.0
Prorations    County taxes 1/1/2025 to 8/25/2025 @ $2,568.37/Year                                   1660.64
              Homeowners Association Dues 8/25/2025 to 1/1/2026 @ $100.00/Year               35.34         
Titel Charges Title - Lender's coverage Premium $0.00 to Texas National Title, Inc.                        
              Title - Settlement or closing fee $1,250.00 to Texas National Title, Inc.      625.0         
              Title - E-Reco

In [85]:
#df.set_index(['Cls','Description']).fillna('')

In [91]:
# Convert to csv
from io import StringIO
with StringIO() as sio:
    df.set_index(['Cls','Description']).fillna('').reset_index().to_csv(sio, index=False)
    csvS = sio.getvalue()
#print(csvS)

In [93]:
# convert to json
import json
import pandas as pd
x = pd.DataFrame(ccDict)
x = df
#print(json.dumps(ccDict, indent=2))
print(x[['Description','Buyer','Seller']].to_json(orient='records', indent=2))

[
  {
    "Description":"Sale Price of Property",
    "Buyer":220000.0,
    "Seller":null
  },
  {
    "Description":"Deposit or Earnest Money from W&B Group, LLC",
    "Buyer":null,
    "Seller":5000.0
  },
  {
    "Description":"Option Money from W&B Group, LLC",
    "Buyer":null,
    "Seller":300.0
  },
  {
    "Description":"County taxes 1\/1\/2025 to 8\/25\/2025 @ $2,568.37\/Year",
    "Buyer":null,
    "Seller":1660.64
  },
  {
    "Description":"Homeowners Association Dues 8\/25\/2025 to 1\/1\/2026 @ $100.00\/Year",
    "Buyer":35.34,
    "Seller":null
  },
  {
    "Description":"Title - Lender's coverage Premium $0.00 to Texas National Title, Inc.",
    "Buyer":null,
    "Seller":null
  },
  {
    "Description":"Title - Settlement or closing fee $1,250.00 to Texas National Title, Inc.",
    "Buyer":625.0,
    "Seller":null
  },
  {
    "Description":"Title - E-Recording Service Fee to Texas National Title, Inc.",
    "Buyer":8.0,
    "Seller":null
  },
  {
    "Description":"Go